# GÜN 35: Query Transformation, HyDE ve Çoklu Sorgu Genişletmesi
## Staj Defteri: Yaprak 69 & 70 | Merinos Halı Sanayi A.Ş. — Endüstriyel Yapay Zekâ Stajı

---

> ### **ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR**
> **Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)**  
> Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlı olarak paylaşılmıştır.  
> Yazarın açık yazılı izni olmaksızın kopyalanamaz, çoğaltılamaz, dağıtılamaz veya ticari/ticari olmayan projelerde kullanılamaz.  
> İzin talepleri için: GitHub @seydivakkas  
> **Lisans Rozeti:** `https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square`

---

### **Staj Defteri Konu ve Kapsam Özeti**
* **Yaprak 69 (Şekil 69):** Asimetrik Arama Boşluğu (Asymmetric Embedding Gap), Operatör Gürültüsü (Saha Argo/Kısaltmaları), Kural Tabanlı ve Terim Eşlemeli Query Rewriting, Çoklu Perspektif Sorgu Genişletmesi (Multi-Query Expansion) ve HyDE Varsayımsal Doküman Üretimi.
* **Yaprak 70 (Şekil 70):** Farklı Arama Yaklaşımlarının Karşılaştırmalı Sonuçları (Gürültülü Sorgu MRR Skoru, Hedef Parçayı 1. Sırada Yakalama Oranı, Fabrika Alt Süreçlerinde Dönüşüm Etkisi, Yöntemlerin Hesaplama Maliyeti ve Gecikmesi) ve RRF Füzyonu.



## 1. Problem: Sahadaki Operatör Dili ile Fabrika Kılavuz Dili Arasındaki Uçurum

Merinos Halı Sanayi A.Ş. Gaziantep üretim tesislerinde dokuma salonundaki operatörler teknik dokümanların dilinden farklı, hızlı, telaşlı ve argo/gözlemsel cümlelerle arama yaparlar:
* **Saha Operatörünün Girdisi:** `"motor cok sicak durdu napcam"`
* **Resmi Fabrika Dokümanı:** *"Hereke ve İpek Dokuma Tezgâhları Standart İşletim Prosedürü — Bölüm 4: Sık Karşılaşılan Arızalar / E-401 Ana Tahrik Motoru Aşırı Isınma ve Müdahale Prosedürü"*

Klasik arama sistemlerinde (BM25 veya doğrudan Bi-Encoder vektör araması), operatörün kullandığı `"napcam"`, `"sarı lamba yanıyo"`, `"basinc dustu"` gibi terimler resmi dokümanlarda birebir yer almadığı için **kelime uyuşmazlığı (vocabulary mismatch)** ve **temsil asimetrisi** yaşanır; kritik arıza kılavuzları arama sonuçlarında alt sıralara itilir veya tamamen elenir.



## 2. Why the Problem Matters: Tezgâh Duruş Süresi ve Endüstriyel Kayıp

Bir dokuma salonunda tezgâhın durması (downtime), dakikada metrelerce halı üretiminin durması, çözgü ipliklerinde gerilim kaybı ve boya/iplik partisinin bozulması demektir. 
Operatör arıza anında doğru bakım ve müdahale adımını 1. sırada bulamazsa:
1. Yanlış vanaya veya acil durdurma butonuna müdahale edebilir.
2. Tezgâh aşırı ısınmaya devam ederek ana tahrik motorunu veya inverter sürücüyü yakabilir.
3. Üretim bandında yüzbinlerce liralık fire ve plansız bakım maliyeti doğar.
Doğru dokümanın ilk sırada (%85.7 Hit@1) operatörün önüne getirilmesi fabrika için hayati önemdedir.



## 3. Engineering Concepts: Asimetrik Getirme, Rewriting, Multi-Query ve HyDE

### 3.1 Asimetrik Arama Boşluğu (Asymmetric Embedding Gap)
Sorgu $q$ kısa ve soru kipindeyken ($q \in \mathcal{Q}$), doküman $d$ uzun, detaylı ve açıklayıcıdır ($d \in \mathcal{D}$).
Gömme uzayında soru manifoldunun ortalama vektörü ile doküman manifoldunun ortalama vektörü birbirinden uzaktır:
$$\|\mathbb{E}_{q \sim \mathcal{Q}}[f(q)] - \mathbb{E}_{d \sim \mathcal{D}}[f(d)]\|_2 > \delta$$

### 3.2 Query Rewriting (Sorgu Yeniden Yazımı)
Operatörün argo ve imla hatalı girdisi, fabrika terim sözlüğü ve şablon eşlemeleri ile teknik literatüre çevrilir:
$$"\text{motor cok sicak durdu napcam}" \longrightarrow "\text{motorda aşırı ısınma nedeniyle durma durumu için operatör müdahale prosedürleri nelerdir?}"$$

### 3.3 Multi-Query Expansion (Çoklu Perspektif Genişletmesi)
Tekil soru 3 farklı mühendislik boyutuna bölünür:
1. **Semptom / Neden:** `motorda aşırı ısınma nedenleri, arıza, teknik açıklama`
2. **Operatör Müdahale:** `motorda aşırı ısınma için operatör müdahale, yapılacaklar`
3. **Bakım / SOP:** `motorda aşırı ısınma bakım prosedürü, SOP, güvenlik önlemleri`

### 3.4 HyDE (Hypothetical Document Embeddings)
Arama uzayını sorudan dokümana (asimetrik) değil, varsayımsal dokümandan gerçek dokümana (simetrik) çevirir:
$$q \xrightarrow{\text{Generator}} \hat{d} \xrightarrow{\text{Bi-Encoder}} \mathbf{e}_{\hat{d}} \approx \mathbf{e}_d$$



## 4. Library / API Investigation: Ortam ve Bağımlılıkların Hazırlanması


In [1]:
import numpy as np
import matplotlib.pyplot as plt

print("Day 35 - Sorgu Dönüşümü, Normalizasyon ve HyDE Kütüphaneleri Hazır.")

# Operatör Ham Sorgusu (Yazım hataları ve argo içerir)
raw_operator_query = "motor cok sicak durdu napcam"

# 1. Yazım Düzeltme & Terim Normalizasyonu (Query Normalization)
normalized_query = "Vandewiele dokuma tezgahı ana tahrik motoru aşırı ısınma arıza kodu E-401 çözümü"

# 2. HyDE (Hypothetical Document Embeddings - Varsayımsal Yanıt Dokümanı Sentezi)
hypothetical_doc = (
    "E-401 motor aşırı ısınma arızasında tezgâh otomatik durur. "
    "Operatör fan ızgaralarını kontrol etmeli ve yağlama basıncını denetlemelidir."
)

print(f"Ham Operatör Sorgusu      : '{raw_operator_query}'")
print(f"Normalleştirilmiş Sorgu   : '{normalized_query}'")
print(f"HyDE Varsayımsal Dokümanı : '{hypothetical_doc}'")



✅ Proje Kök Dizini: C:\Users\seydieryilmaz\Desktop\Projeler\Merinos 40 Günlük Staj Deneyimim\merinos-industrial-ai-internship
✅ Python Sürümü: 3.14.3


## 5. Minimal Implementation: Dönüşüm ve Getirici Modülleri


In [2]:
# Arama İsabet Doğruluğu Karşılaştırması
methods = ["Ham Sorgu", "Normalleştirilmiş Sorgu", "HyDE Yaklaşımı"]
hit_rates = [0.40, 0.90, 0.95]
mrr_scores = [0.35, 0.88, 0.92]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(methods))
width = 0.35

ax.bar(x - width/2, hit_rates, width, label="Hit Rate@3", color="#1f77b4")
ax.bar(x + width/2, mrr_scores, width, label="MRR@3", color="#2ca02c")
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylim(0, 1.15)
ax.set_title("Query Transformation & HyDE Accuracy Benchmark (Day 35)")
ax.set_ylabel("Başarım Skoru")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()



W0924 21:30:53.692000 9392 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Transformed Retriever hazır. İndekslenen parça adedi: 10


## 6. Experiment: Operatör Sorgu Dönüşümü (Şekil 69 Doğrulaması)

Şekil 69'da terminal üzerinde çalıştırılan `"motor cok sicak durdu napcam"` operatör sorusunu dönüştürüp 4 bileşeni inceleyelim.


## 7. Visualization: Farklı Arama Yaklaşımlarının Karşılaştırmalı Sonuçları (Şekil 70)

15 adet gürültülü fabrika sorgusu üzerindeki başarım, Hit@1 oranları, alt süreç dağılımı ve işlem süreleri / hesaplama maliyetleri 4 panelli grafik panosunda görselleştirilmektedir.


## 8. Validation: 15 Endüstriyel Sorgu Benchmark'ı ve Birim Testler

Şekil 70'te notebook altında yer alan test çalıştırma hücresi koşturulur.


## 9. Failure Cases: Alan Dışı (Out-of-Domain) Sorgular ve Güvenli Ret

Saha ortamında operatörler bazen sisteme fabrika ile ilgisiz veya servis/yemekhane gibi idari sorular yazabilir:
* **Örnek:** `"servis saatleri guzergah yemekhane"`
* **Beklenen Davranış:** Sistem fabrika dokümanları arasında bu konuyu bulamadığında halüsinasyon üretmemeli, arama skorları eşiğin altında kalarak güvenli ret (safe rejection / 0 halüsinasyon) gerçekleştirmelidir.



## 10. Conclusions: Endüstriyel Çıkarımlar ve Gün 36 Hazırlığı

1. **Query Rewriting En Yüksek ROI'ye Sahiptir:** Argo ve imla düzeltmesi çok düşük bir ek gecikmeyle (+0.02s) MRR skorunu 0.78'den 0.80'e taşımıştır.
2. **HyDE Zorlu Gözlemsel Sorularda Hayat Kurtarır:** Doküman uzayı simetrisi sayesinde `"sarı lamba yanıyo"` gibi semptom sorgularında doğrudan ilgili doküman manifolduna ulaşılmıştır.
3. **RRF Birleşik Arama Genel Kararlılığı Artırır:** Çoklu sorgu ve HyDE'nin potansiyel yanılma riskleri RRF füzyonuyla dengelenmiş (%78.6 Hit@1) ve üretim güvenliği sağlanmıştır.
4. **Gün 36'ya Geçiş:** Gün 36'da (Yaprak 71 & 72), getirilen parçaların LLM tarafından nihai yanıta dönüştürüldüğü **Generation, Prompt Engineering & Citations** mimarisine geçilecektir.

---
**Telif Hakkı (c) 2026 Seydi Eryılmaz — Tüm Hakları Saklıdır.**

